# 0. import libraries

In [1]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# ── 2. circuit_tracer model for attribution (replaces HF model entirely) ─────
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend="transformerlens"
)

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


# 1. getting attributes from each token step
based on attribute_demo  
**can skip to step 2, features are saved in** `demos\graph_files`

In [ ]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 8192  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 256  # Batch size when attributing
offload = (
    "disk" if IN_COLAB else "cpu"
)  # Offload various parts of the model during attribution to save memory. Can be 'disk', 'cpu', or None (keep on GPU)
verbose = True  # Whether to display a tqdm progress bar and timing report

: 

In [ ]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe stared into the orange sun,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=max_n_logits,
        desired_logit_prob=desired_logit_prob,
        max_feature_nodes=max_feature_nodes,   # ← missing
        batch_size=batch_size,
        offload=offload,                        # ← use the variable, not hardcoded "cpu"
        verbose=verbose,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files/gemma-2-2b-2/",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

Phase 0: Precomputing activations and vectors


Generated: And saw the world in a new way


Precomputation completed in 1.51s
Found 13259 active features
Phase 1: Running forward pass
Forward pass completed in 14.99s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5664
Will include 8192 of 13259 feature nodes
Input vectors built in 0.62s
Phase 3: Computing logit attributions
c:\Users\sirichada\Github\circuit-tracer\.venv\Lib\site-packages\circuit_tracer\attribution\context_transformerlens.py:223: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  self._resid_activations[last_layer].backward(
Logit attributions completed in 4.64s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [02:02<00:00, 67.08it/s]
Feature attributions completed in 122.13s
Attribution completed in 144.00s


✓ step 00 → 'And'  saved as 'step-00-And'


Precomputation completed in 0.79s
Found 14075 active features
Phase 1: Running forward pass
Forward pass completed in 17.18s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5000
Will include 8192 of 14075 feature nodes
Input vectors built in 0.25s
Phase 3: Computing logit attributions
Logit attributions completed in 4.57s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [02:10<00:00, 62.56it/s]
Feature attributions completed in 130.95s
Attribution completed in 153.83s
Phase 0: Precomputing activations and vectors


...

Precomputation completed in 0.68s
Found 16971 active features
Phase 1: Running forward pass
Forward pass completed in 20.55s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.4980
Will include 8192 of 16971 feature nodes
Input vectors built in 0.20s
Phase 3: Computing logit attributions
Logit attributions completed in 5.47s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [02:31<00:00, 54.20it/s]
Feature attributions completed in 151.14s
Attribution completed in 178.11s
Phase 0: Precomputing activations and vectors


✓ step 04 → ' in'  saved as 'step-04-in'


Precomputation completed in 0.70s
Found 17837 active features
Phase 1: Running forward pass
Forward pass completed in 21.02s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.5156
Will include 8192 of 17837 feature nodes
Input vectors built in 0.20s
Phase 3: Computing logit attributions
Logit attributions completed in 5.59s
Phase 4: Computing feature attributions
Feature influence computation: 100%|██████████| 8192/8192 [02:45<00:00, 49.43it/s]
Feature attributions completed in 165.73s
Attribution completed in 193.32s
Phase 0: Precomputing activations and vectors


✓ step 05 → ' a'  saved as 'step-05-a'


Precomputation completed in 0.64s
Found 18536 active features
Phase 1: Running forward pass
Forward pass completed in 22.13s
Phase 2: Building input vectors
Using 10 salient logits with cumulative probability 0.4766
Will include 8192 of 18536 feature nodes
Input vectors built in 0.20s
Phase 3: Computing logit attributions


In [ ]:
# extracting clerp descriptions and adding to graph files

import json, time, requests
from pathlib import Path

def fetch_clerp(layer, feature):
    url = f"https://www.neuronpedia.org/api/feature/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feature}"
    r = requests.get(url)
    if r.ok:
        data = r.json()
        descs = data.get("explanations", [])
        if descs:
            return descs[0].get("description", "")
    return ""

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        parts = node['node_id'].split('_')
        feat = parts[1]
        layer = node['layer']
        node['clerp'] = fetch_clerp(layer, feat)
        time.sleep(0.1)  # be nice to the API
    json_path.write_text(json.dumps(data))
    print(f"done: {json_path.name}")

# 2. attribute graph visualization

In [1]:
import json
from pathlib import Path

graph_dir = Path("./graph_files/gemma-2-2b-2")

for json_path in sorted(graph_dir.glob("step-*.json")):
    data = json.loads(json_path.read_text())
    
    # collect all unique layers used by transcoder nodes
    layers = sorted(set(
        int(node["layer"])
        for node in data["nodes"]
        if not node["is_target_logit"] and node.get("feature_type") == "cross layer transcoder"
    ))
    
    transcoder_list = [f"gemma-2-2b/{l}-gemmascope-transcoder-16k" for l in layers]
    data["metadata"]["transcoder_list"] = transcoder_list
    
    json_path.write_text(json.dumps(data))
    print(f"{json_path.name}: {len(layers)} layers → {transcoder_list[:3]}...")

# also patch the manifest
manifest_path = graph_dir / "graph-metadata.json"
manifest = json.loads(manifest_path.read_text())
for entry in manifest["graphs"]:
    slug = entry["slug"]
    step_path = graph_dir / f"{slug}.json"
    if step_path.exists():
        step_data = json.loads(step_path.read_text())
        entry["transcoder_list"] = step_data["metadata"]["transcoder_list"]

manifest_path.write_text(json.dumps(manifest, indent=2))
print("manifest patched")

step-00-And.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-01-saw.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-02-the.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-03-world.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-04-in.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
step-05-a.json: 26 layers → ['gemma-2-2b/0-gemmascope-transcoder-16k', 'gemma-2-2b/1-gemmascope-transcoder-16k', 'gemma-2-2b/2-gemmascope-transcoder-16k']...
manifest patched


In [4]:
import json
from pathlib import Path

data = {"graphs": []}  # ← Add this line to initialize

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    graph_data = json.loads(json_path.read_text())
    data["graphs"].append(graph_data["metadata"])

Path("./graph_files/gemma-2-2b-2/graph-metadata.json").write_text(json.dumps(data, indent=2))
print("done")

done


In [5]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/gemma-2-2b-2/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

http://localhost:8046/index.html


## 2.1 feature analysis based on circuit

In [2]:
import json
from pathlib import Path

def show_top_nodes(json_path, n=10):
    data = json.loads(Path(json_path).read_text())
    slug = data['metadata']['slug']
    
    # filter out logit nodes, sort by influence
    nodes = [n for n in data['nodes'] if not n['is_target_logit'] and n['influence'] is not None]
    nodes = sorted(nodes, key=lambda x: x['influence'], reverse=True)[:n]
    
    print(f"\n=== {slug} ===")
    for node in nodes:
        layer = node['layer']
        node_id_parts = node['node_id'].split('_')
        feat = node_id_parts[1]
        influence = node['influence']
        url = f"https://www.neuronpedia.org/gemma-2-2b/{layer}-gemmascope-transcoder-16k/{feat}"
        print(f"  L{layer}F{feat:>6}  influence={influence:.4f}  {url}")

for f in sorted(Path("./graph_files/gemma-2-2b-2").glob("*.json")):
    show_top_nodes(f)

KeyError: 'metadata'

In [6]:
import json
from pathlib import Path
from collections import defaultdict

position_features = defaultdict(list)

for json_path in sorted(Path("./graph_files/gemma-2-2b-2").glob("step-*.json")):
    data = json.loads(json_path.read_text())
    for node in data['nodes']:
        if node['is_target_logit'] or node['influence'] is None:
            continue
        token = data['metadata']['prompt_tokens'][node['ctx_idx']]
        if token == '\n':
            parts = node['node_id'].split('_')
            key = f"L{node['layer']}F{parts[1]}"
            position_features[key].append((json_path.stem, node['influence'], node['clerp']))

# show features appearing in 3+ steps, sorted by occurrence count
for feat, occurrences in sorted(position_features.items(), key=lambda x: -len(x[1])):
    if len(occurrences) >= 3:
        clerp = occurrences[0][2] or "no description"
        print(f"{feat}  ({len(occurrences)}/9 steps)  clerp: {clerp}")
        for step, influence, _ in occurrences:
            print(f"    {step}  influence={influence:.4f}")

L0F3259  (12/9 steps)  clerp: no description
    step-00-And  influence=0.5777
    step-00-And  influence=0.4057
    step-01-saw  influence=0.6031
    step-01-saw  influence=0.6652
    step-02-the  influence=0.6213
    step-02-the  influence=0.6449
    step-03-world  influence=0.6212
    step-03-world  influence=0.7559
    step-04-in  influence=0.6235
    step-04-in  influence=0.6854
    step-05-a  influence=0.6284
    step-05-a  influence=0.6935
L0F3850  (12/9 steps)  clerp: no description
    step-00-And  influence=0.4075
    step-00-And  influence=0.3826
    step-01-saw  influence=0.3898
    step-01-saw  influence=0.4972
    step-02-the  influence=0.4305
    step-02-the  influence=0.4425
    step-03-world  influence=0.4333
    step-03-world  influence=0.5106
    step-04-in  influence=0.4488
    step-04-in  influence=0.4188
    step-05-a  influence=0.4370
    step-05-a  influence=0.4307
L0F6625  (12/9 steps)  clerp: no description
    step-00-And  influence=0.7197
    step-00-And  in

**Most promising (identified by Claude Sonnet 4.6):**

L2F629 — "marks verses, choruses, or segments of a musical composition" — active on \n across all 9 steps, consistently  
L8F13615 — "words and phrases related to poetry or creative writing with a bias toward personal anecdotes" — 17/9 steps  
L4F4619 — "lyrics of a specific song, possibly about love and uncertainty" — 17/9 steps  
L5F6651 — "sad and inquisitive poetry" — 9/9 steps, consistent influence ~0.6-0.78  
L8F12808 — "lines of poetry, especially those that express questioning, prayer, or lament" — 9/9 steps  
L13F4053 — "lines of rap lyrics" — 12/9 steps, high influence  
L3F14659 — "the language of rap lyrics" — 18/9 steps  
L14F15021 — "phrases that begin a line of poetry" — this one is particularly interesting given the context  
L9F11782 — "instances of the word 'poem' or descriptive phrases about things, often with a sense of place" — 5/9 steps  
L8F13396 — "mentions of poetic or musical art forms" — 5/9 steps, hits 0.8000 influence at step-07

# 3. intervention
based on intervention_demo

In [ ]:
from collections import namedtuple
from functools import partial

import torch 

from circuit_tracer import ReplacementModel

# display functions
from circuit_tracer.utils.demo_utils import display_topk_token_predictions, display_generations_comparison

backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
model = ReplacementModel.from_pretrained("google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend=backend)


Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded pretrained model google/gemma-2-2b into HookedTransformer


In [ ]:
Feature = namedtuple('Feature', ['layer', 'pos', 'feature_idx'])

# a display function that needs the model's tokenizer
display_topk_token_predictions = partial(display_topk_token_predictions, tokenizer=model.tokenizer)

In [ ]:
# L14F15021 = "phrases that begin a line of poetry"
supernode_features = [
    Feature(layer=14,pos=-1,feature_idx=15021),
]

intervention_tuples = [(*supernode_feature, 0.0) for supernode_feature in supernode_features]

In [ ]:
s = "A rhyming couplet:\nHe stared into the orange sun,\n"

In [ ]:
with torch.inference_mode():
    original_logits, _  = model.feature_intervention(s, [])
    new_logits, _ = model.feature_intervention(s, intervention_tuples)

display_topk_token_predictions(s, original_logits, new_logits)

Token,Probability,Distribution
He,0.152,15.2%
But,0.152,15.2%
And,0.072,7.2%
The,0.044,4.4%
Then,0.039,3.9%
Token,Probability,Distribution
He,0.155,15.5%
But,0.137,13.7%
And,0.073,7.3%
but,0.039,3.9%


In [ ]:
from circuit_tracer.utils.demo_utils import display_generations_comparison

# suppress both candidate features at the newline position (-1)
intervention_tuples = [
    (14, -1, 15021, 0.0), # L14F15021 - phrases that begin a line of poetry
]

pre = [model.feature_intervention_generate(s, [], do_sample=False, verbose=False)[0]]
post = [model.feature_intervention_generate(s, intervention_tuples, do_sample=False, verbose=False)[0]]

display_generations_comparison(s, pre, post)